# 13 Vetting Score

Composite Vetting Score (mimics real exoplanet vetting pipelines)
=====================================================================
Suggested notebook name: 13_vetting_score.ipynb (place in notebooks/)

WHY: Reproduces the named, transparent checks real pipelines (Kepler
Robovetter, TESS QLP) run -- independent of the ML model entirely.

IMPORTANT NAMING FIX: the old version's verdict field used the value
"likely_false_positive" -- now that the ML model has an actual class
literally called "false_positive", that name collision would be
confusing (the two are computed completely independently and can
disagree). Renamed the verdict column to rule_based_verdict and the
value to likely_false_positive_by_rules to make clear this is a
SEPARATE, independent check -- compare it against confidence_triage.csv's
ML predictions rather than treating them as the same thing.

Checks implemented:
    1. SNR check               -- signal strong enough to trust at all
    2. Odd-Even check          -- transit depth consistent between odd/even
                                   transits (large mismatch => eclipsing binary)
    3. Secondary eclipse check -- presence of a secondary dip (=> EB, not planet)
    4. Transit shape check     -- skewness close to flat-bottomed (planet-like)
                                   vs V-shaped (EB-like)

NOTE ON THRESHOLDS: defaults below are reasonable starting points, NOT
calibrated to your specific data. Run against labeled_features.csv first
(known planets/EBs/FPs) to sanity-check before trusting it on science stars.

In [1]:
import pandas as pd

ID_COL = "tic_id"
DATA_PATH = "../outputs/features.csv"
OUTPUT_PATH = "../outputs/vetting_report.csv"

SNR_MIN = 7.0
ODD_EVEN_MAX = 0.01
SECONDARY_DEPTH_MAX = 0.0005
SKEW_ABS_MAX = 1.0

df = pd.read_csv(DATA_PATH)
df["tic_id"] = df["tic_id"].astype(str).str.replace(".0", "", regex=False)

required = ["snr", "odd_even_diff", "secondary_depth", "transit_skewness"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {DATA_PATH}: {missing}\nAvailable: {list(df.columns)}")


def vet_row(row):
    checks = {
        "snr_check_pass": bool(row["snr"] >= SNR_MIN),
        "odd_even_check_pass": bool(row["odd_even_diff"] <= ODD_EVEN_MAX),
        "secondary_eclipse_check_pass": bool(row["secondary_depth"] <= SECONDARY_DEPTH_MAX),
        "transit_shape_check_pass": bool(abs(row["transit_skewness"]) <= SKEW_ABS_MAX),
    }
    score = sum(checks.values())
    checks["vetting_score"] = f"{score}/4"
    checks["rule_based_verdict"] = (
        "strong_candidate" if score == 4 else
        "weak_candidate" if score >= 2 else
        "likely_false_positive_by_rules"
    )
    return checks


vetting_results = df.apply(vet_row, axis=1, result_type="expand")
report = pd.concat([df[[ID_COL]], vetting_results], axis=1)
report.to_csv(OUTPUT_PATH, index=False)

print(report.to_string(index=False))
print(f"\nSaved -> {OUTPUT_PATH}")
print(f"\nVerdict counts:\n{report['rule_based_verdict'].value_counts().to_string()}")
print("\nThis is an INDEPENDENT rule-based check, separate from the ML model's")
print("predictions. Compare against confidence_triage.csv -- agreement between")
print("the two is your strongest evidence; disagreement is worth investigating.")

   tic_id  snr_check_pass  odd_even_check_pass  secondary_eclipse_check_pass  transit_shape_check_pass vetting_score rule_based_verdict
149603524           False                 True                          True                      True           3/4     weak_candidate
229742722           False                 True                          True                      True           3/4     weak_candidate
237201858           False                 True                          True                      True           3/4     weak_candidate
260647166            True                 True                          True                      True           4/4   strong_candidate
261136641           False                 True                          True                      True           3/4     weak_candidate
261136679            True                 True                          True                      True           4/4   strong_candidate
261136765           False                 True  

In [8]:
import pandas as pd
import numpy as np
import os, glob

print('=' * 55)
print('  EXOPLANET PIPELINE — BUG AUDIT REPORT')
print('=' * 55)

# 1. Check all output files exist
print('\n1. OUTPUT FILES')
files = {
    'bls_all_results.csv'      : '../outputs/bls_all_results.csv',
    'features.csv'             : '../outputs/features.csv',
    'labeled_features.csv'     : '../outputs/labeled_features.csv',
    'results.csv'              : '../outputs/results.csv',
    'final_classification.csv' : '../outputs/final_classification.csv',
    'transit_params.csv'       : '../outputs/transit_params.csv',
    'planet_parameters.csv'    : '../outputs/planet_parameters.csv',
    'injection_recovery.csv'   : '../outputs/injection_recovery.csv',
}
for name, path in files.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists else 0
    print(f'  {"✅" if exists and size>10 else "❌"} {name:35s} {size:>8} bytes')

# 2. Check for corrupted star
print('\n2. CORRUPTED STAR CHECK (TIC 261203535)')
for name, path in files.items():
    if not os.path.exists(path): continue
    try:
        df = pd.read_csv(path)
        if 'tic_id' in df.columns:
            present = df['tic_id'].astype(str).str.contains('261203535').any()
            print(f'  {"❌ PRESENT" if present else "✅ excluded"} in {name}')
    except: pass

# 3. Check SNR values
print('\n3. SNR SANITY CHECK')
if os.path.exists('../outputs/features.csv'):
    feat = pd.read_csv('../outputs/features.csv')
    feat['tic_id'] = feat['tic_id'].astype(str).str.replace('.0','')
    print(f'  SNR range     : {feat["snr"].min():.3f} to {feat["snr"].max():.3f}')
    print(f'  SNR > 1000    : {(feat["snr"] > 1000).sum()} stars (should be 0)')
    print(f'  depth > 1e6   : {(feat["depth_ppm"] > 1e6).sum()} stars (should be 0)')
    if 'bls_snr_raw' in feat.columns:
        print(f'  bls_snr_raw   : {feat["bls_snr_raw"].min():.3f} to {feat["bls_snr_raw"].max():.3f}')
    else:
        print('  ⚠️  bls_snr_raw column missing — SNR normalization not applied')

# 4. Check labeled features
print('\n4. LABELED FEATURES CHECK')
if os.path.exists('../outputs/labeled_features.csv'):
    lab = pd.read_csv('../outputs/labeled_features.csv')
    print(f'  Total samples : {len(lab)}')
    if 'label' in lab.columns:
        print(f'  Label counts  : {lab["label"].value_counts().to_dict()}')
    print(f'  NaN count     : {lab.isnull().sum().sum()} total NaNs')
    for col in ['snr','depth_ppm','cdpp_2h']:
        if col in lab.columns:
            nans = lab[col].isna().sum()
            print(f'  NaN in {col:12s}: {nans}')

# 5. Check model files
print('\n5. MODEL FILES')
models = ['random_forest.pkl','xgboost_model.pkl',
          'label_encoder.pkl','feature_cols.pkl']
for m in models:
    path = f'../models/{m}'
    print(f'  {"✅" if os.path.exists(path) else "❌"} {m}')

if os.path.exists('../models/label_encoder.pkl'):
    import pickle
    with open('../models/label_encoder.pkl','rb') as f:
        le = pickle.load(f)
    print(f'  Classes: {le.classes_}')

# 6. Check transit params
print('\n6. TRANSIT PARAMS RELIABILITY')
if os.path.exists('../outputs/transit_params.csv'):
    tp = pd.read_csv('../outputs/transit_params.csv')
    print(f'  Total rows    : {len(tp)}')
    for col in tp.columns:
        nans = tp[col].isna().sum()
        if nans > 0:
            print(f'  ⚠️  NaN in {col}: {nans}')
    if 'fit_reliable' in tp.columns:
        print(f'  Reliable fits : {tp["fit_reliable"].sum()} / {len(tp)}')
    else:
        print('  ⚠️  fit_reliable column missing')
    if 'depth_ppm' in tp.columns:
        bad = tp[tp['depth_ppm'] > 1e6] if 'depth_ppm' in tp.columns else pd.DataFrame()
        print(f'  Bad depth fits: {len(bad)} (depth > 1e6 ppm)')

# 7. Check classification results
print('\n7. CLASSIFICATION RESULTS')
if os.path.exists('../outputs/final_classification.csv'):
    fc = pd.read_csv('../outputs/final_classification.csv')
    fc['tic_id'] = fc['tic_id'].astype(str).str.replace('.0','')
    print(f'  Total stars   : {len(fc)}')
    if 'ml_label' in fc.columns:
        print(f'  Labels        : {fc["ml_label"].value_counts().to_dict()}')
    if 'ml_confidence' in fc.columns:
        print(f'  Conf range    : {fc["ml_confidence"].min():.2f} to {fc["ml_confidence"].max():.2f}')

# 8. Injection recovery
print('\n8. INJECTION RECOVERY')
if os.path.exists('../outputs/injection_recovery.csv'):
    ir = pd.read_csv('../outputs/injection_recovery.csv')
    print(f'  Total tests   : {len(ir)}')
    print(f'  Recovery rate : {ir["recovered"].mean()*100:.1f}%')
else:
    print('  ❌ Not run yet')

print('\n' + '=' * 55)
print('  END OF AUDIT')
print('=' * 55)

  EXOPLANET PIPELINE — BUG AUDIT REPORT

1. OUTPUT FILES
  ✅ bls_all_results.csv                     4572 bytes
  ✅ features.csv                           10649 bytes
  ✅ labeled_features.csv                    9296 bytes
  ✅ results.csv                            11242 bytes
  ✅ final_classification.csv               10664 bytes
  ✅ transit_params.csv                      1367 bytes
  ✅ planet_parameters.csv                    807 bytes
  ✅ injection_recovery.csv                 37724 bytes

2. CORRUPTED STAR CHECK (TIC 261203535)
  ✅ excluded in bls_all_results.csv
  ✅ excluded in features.csv
  ✅ excluded in labeled_features.csv
  ✅ excluded in results.csv
  ✅ excluded in final_classification.csv
  ✅ excluded in transit_params.csv
  ✅ excluded in planet_parameters.csv
  ✅ excluded in injection_recovery.csv

3. SNR SANITY CHECK
  SNR range     : 0.000 to 1.000
  SNR > 1000    : 0 stars (should be 0)
  depth > 1e6   : 0 stars (should be 0)
  bls_snr_raw   : 3.825 to 35.553

4. LABELED